# Extra-Channel Prediction Context

Inspect how the 5-channel models predict when channel 4 and channel 5 are available as inputs.

## Setup

In [ ]:
from pathlib import Path
import sys

import torch
from IPython.display import Markdown, display

CWD = Path.cwd()
if (CWD / "src").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "src").exists():
    REPO_ROOT = CWD.parent
else:
    raise RuntimeError("Could not find repo root containing src/.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_dataloaders, load_class_map
from src.evaluate import evaluate_test_metrics, load_checkpoint_model
from src.train import sync_class_config
from src.utils import get_device, load_config
from src.visualize import plot_extra_channel_context_grid

device = get_device()
device

## Load Config

In [ ]:
comparison_config = load_config(REPO_ROOT / "configs" / "channel_comparison.yaml")
viz_config = comparison_config["visualization"]
class_map = load_class_map(REPO_ROOT / comparison_config["data"]["class_map"])
class_names = class_map["class_names"]
ignore_index = class_map.get("ignore_index", 255)
output_dir = REPO_ROOT / viz_config.get("output_dir", "notebooks/outputs/channel_comparison/")
output_dir.mkdir(parents=True, exist_ok=True)

comparison_config["models"]

## Select Test Samples

In [ ]:
base_entry = comparison_config["models"][0]
base_train_config = sync_class_config(load_config(REPO_ROOT / base_entry["config"]))
base_train_config["data"]["channels"] = list(range(1, base_entry["in_channels"] + 1))
base_train_config["data"]["num_classes"] = base_entry["num_classes"]

_, _, test_loader = build_dataloaders(base_train_config["data"])
test_dataset = test_loader.dataset

sample_indices = viz_config.get("sample_indices", list(range(viz_config.get("num_samples", 6))))
sample_indices = [index for index in sample_indices if index < len(test_dataset)]
samples = [test_dataset[index] for index in sample_indices]
sample_images = torch.stack([sample["image"] for sample in samples])
sample_masks = torch.stack([sample["mask"] for sample in samples])

len(samples), sample_images.shape, sample_masks.shape

## Helpers

In [ ]:
def load_five_channel_model(entry):
    train_config = sync_class_config(load_config(REPO_ROOT / entry["config"]))
    train_config["model"]["in_channels"] = entry["in_channels"]
    train_config["model"]["num_classes"] = entry["num_classes"]
    train_config["data"]["channels"] = list(range(1, entry["in_channels"] + 1))
    model, checkpoint = load_checkpoint_model(
        REPO_ROOT / entry["checkpoint"],
        device=device,
        model_config=train_config["model"],
    )
    return model, checkpoint


@torch.no_grad()
def predict_samples(model, images):
    logits = model(images.to(device))
    return logits.argmax(dim=1).cpu()

## Extra-Channel Ablation Metrics

In [ ]:
ablation_results = []
for entry in comparison_config["models"]:
    model, checkpoint = load_five_channel_model(entry)
    full_metrics = None
    for ablation in comparison_config["ablations"]:
        print(f"Evaluating {entry['label']} - {ablation['label']}")
        metrics = evaluate_test_metrics(
            model=model,
            dataloader=test_loader,
            num_classes=entry["num_classes"],
            device=device,
            ignore_index=ignore_index,
            class_names=class_names,
            input_transform=zero_channel_transform(ablation["zero_channels"]),
            desc=f"{entry['name']} / {ablation['name']}",
        )
        if ablation["name"] == "full_5ch":
            full_metrics = metrics
        ablation_results.append({
            "model": entry["label"],
            "ablation": ablation["label"],
            "mIoU": metrics["mIoU"],
            "delta_mIoU": metrics["mIoU"] - full_metrics["mIoU"] if full_metrics else 0.0,
            "pixel_accuracy": metrics["pixel_accuracy"],
            "delta_pixel_accuracy": metrics["pixel_accuracy"] - full_metrics["pixel_accuracy"] if full_metrics else 0.0,
            "mean_dice": metrics["mean_dice"],
            "delta_mean_dice": metrics["mean_dice"] - full_metrics["mean_dice"] if full_metrics else 0.0,
            "mAP": metrics["mAP"],
            "delta_mAP": metrics["mAP"] - full_metrics["mAP"] if full_metrics else 0.0,
        })

markdown_table(
    ablation_results,
    [
        ("model", "Model"),
        ("ablation", "Input"),
        ("mIoU", "mIoU"),
        ("delta_mIoU", "Delta mIoU"),
        ("pixel_accuracy", "Pixel Acc"),
        ("delta_pixel_accuracy", "Delta Pixel Acc"),
        ("mean_dice", "Mean Dice"),
        ("delta_mean_dice", "Delta Mean Dice"),
        ("mAP", "mAP"),
        ("delta_mAP", "Delta mAP"),
    ],
)

## Qualitative Ablation: Full 5ch vs Channels 4+5 Zeroed

In [ ]:
rgb_only = next(ablation for ablation in comparison_config["ablations"] if ablation["name"] == "rgb_only")
for entry in comparison_config["models"]:
    display(Markdown(f"### {entry['label']}"))
    model, checkpoint = load_five_channel_model(entry)
    full_predictions = predict_samples(model, sample_images)
    rgb_only_predictions = predict_samples(model, sample_images, zero_channels=rgb_only["zero_channels"])

    fig = plot_extra_channel_context_grid(
        images=sample_images,
        masks=sample_masks,
        predictions=full_predictions,
        extra_channels=viz_config["extra_channels"],
        max_items=viz_config.get("num_samples", 6),
        rgb_channels=tuple(viz_config.get("rgb_channels", [0, 1, 2])),
        class_names=class_names,
        ignore_index=ignore_index,
        title=f"{entry['label']} - full 5ch input",
    )
    fig.savefig(output_dir / f"{entry['name']}_full_5ch_context.png", dpi=viz_config.get("dpi", 150), bbox_inches="tight")
    display(fig)

    fig = plot_extra_channel_context_grid(
        images=sample_images,
        masks=sample_masks,
        predictions=rgb_only_predictions,
        extra_channels=viz_config["extra_channels"],
        max_items=viz_config.get("num_samples", 6),
        rgb_channels=tuple(viz_config.get("rgb_channels", [0, 1, 2])),
        class_names=class_names,
        ignore_index=ignore_index,
        title=f"{entry['label']} - channels 4+5 zeroed",
    )
    fig.savefig(output_dir / f"{entry['name']}_rgb_only_ablation_context.png", dpi=viz_config.get("dpi", 150), bbox_inches="tight")
    display(fig)

## Extra-Channel Context Grids

In [ ]:
for entry in comparison_config["models"]:
    display(Markdown(f"### {entry['label']}"))
    model, checkpoint = load_five_channel_model(entry)
    predictions = predict_samples(model, sample_images)
    fig = plot_extra_channel_context_grid(
        images=sample_images,
        masks=sample_masks,
        predictions=predictions,
        extra_channels=viz_config["extra_channels"],
        max_items=viz_config.get("num_samples", 6),
        rgb_channels=tuple(viz_config.get("rgb_channels", [0, 1, 2])),
        class_names=class_names,
        ignore_index=ignore_index,
        title=f"{entry['label']} extra-channel prediction context",
    )
    fig.savefig(output_dir / f"{entry['name']}_extra_channel_context.png", dpi=viz_config.get("dpi", 150), bbox_inches="tight")
    display(fig)